# Partie 1 — Web Scraping France Info
**Projet Algorithmique Python S2 — Master 1 IdL 2025/2026**

Source : **francetvinfo.fr** (sections autorisees par robots.txt)

Objectif : corpus de **50 articles** (5 categories x 10)

Colonnes : `url | titre | date | chapeau | texte | label`

---
### Correction extraction texte
**Probleme** : `//p//text()` extrayait menus + newsletters + videos (contenu identique sur chaque article).

**Solution** : `//article//p//text()` cible uniquement le corps de l'article.

```
Page HTML
├── <header>   menus navigation     (IGNORE)
├── <article>  corps de l'article   (ON PREND CA)
├── <aside>    videos recommandees  (IGNORE)
└── <footer>   newsletters, liens   (IGNORE)
```

In [ ]:
import sys
!{sys.executable} -m pip install requests lxml pandas --quiet

In [ ]:
import requests
from lxml import html
import csv
import time
import os
import pandas as pd

SECTIONS = {
    'politique': 'https://www.francetvinfo.fr/politique/',
    'sport':     'https://www.francetvinfo.fr/sports/',
    'economie':  'https://www.francetvinfo.fr/economie/',
    'sante':     'https://www.francetvinfo.fr/sante/',
    'sciences':  'https://www.francetvinfo.fr/sciences/',
}

NB_ARTICLES = 10
FICHIER_CSV = 'articles.csv'
HEADERS = {'User-Agent': 'Mozilla/5.0'}

print('Librairies importees avec succes.')

In [ ]:
def extraire_label_depuis_url(url):
    """
    Extrait le label depuis l'URL.
    Entree : url (str)
    Sortie : label (str) ou None
    """
    parties = url.replace('https://www.francetvinfo.fr/', '').split('/')
    for label in SECTIONS.keys():
        if label in parties or (label == 'sport' and 'sports' in parties):
            return label
    return None

print(extraire_label_depuis_url('https://www.francetvinfo.fr/politique/article.html'))
print(extraire_label_depuis_url('https://www.francetvinfo.fr/sports/article.html'))

In [ ]:
def extraire_metadonnees(tree, url):
    """
    Extrait les metadonnees d'un article.
    Entree : tree (lxml.html), url (str)
    Sortie : dict {titre, date, chapeau, label}
    """
    titre = ''
    h1 = tree.xpath('//h1//text()')
    if h1:
        titre = h1[0].strip()
    else:
        title_tag = tree.xpath('//title//text()')
        if title_tag:
            titre = title_tag[0].strip()

    date = ''
    time_tag = tree.xpath('//time/@datetime')
    if time_tag:
        date = time_tag[0].strip()
    else:
        meta_date = tree.xpath("//meta[@property='article:published_time']/@content")
        if meta_date:
            date = meta_date[0].strip()

    chapeau = ''
    meta_desc = tree.xpath("//meta[@name='description']/@content")
    if meta_desc:
        chapeau = meta_desc[0].strip()

    label = extraire_label_depuis_url(url)
    return {'titre': titre, 'date': date, 'chapeau': chapeau, 'label': label}

print('Fonction extraire_metadonnees definie.')

## Fonction cle : extraction du texte

Correction appliquee : `//article//p//text()` au lieu de `//p//text()`

In [ ]:
def extraire_texte_article(tree):
    """
    Extrait uniquement le texte du corps de l'article.

    CORRECTION : on cible <article> pour eviter les menus et
    contenus repetes (newsletters, videos, footer...).

    Strategie en cascade :
      1. //article//p//text()  - balise article
      2. //div[@class='...']   - classes CSS
      3. //p//text()           - fallback global

    Entree : tree (lxml.html)
    Sortie : texte (str) nettoye
    Complexite : O(n) ou n = nombre de noeuds HTML
    """
    paragraphes = tree.xpath('//article//p//text()')

    if not paragraphes:
        paragraphes = tree.xpath(
            "//*[contains(@class,'article') or contains(@class,'content')"
            " or contains(@class,'body') or contains(@class,'main')]//p//text()"
        )

    if not paragraphes:
        paragraphes = tree.xpath('//p//text()')

    texte_brut = '\n'.join(paragraphes)
    lignes_propres = []
    for ligne in texte_brut.splitlines():
        ligne_nettoyee = ligne.strip()
        if ligne_nettoyee and len(ligne_nettoyee) > 20:
            lignes_propres.append(ligne_nettoyee)

    return '\n'.join(lignes_propres)

print('Fonction extraire_texte_article definie.')

In [ ]:
def scraper_section(url_section, label, nb_articles=10):
    """
    Recupere les articles d'une section France Info.
    Entree : url_section (str), label (str), nb_articles (int)
    Sortie : liste de dicts {url, titre, date, chapeau, texte, label}
    """
    resultats = []
    try:
        reponse = requests.get(url_section, headers=HEADERS, timeout=10)
        reponse.raise_for_status()
    except Exception as e:
        print('Erreur section ' + label + ' : ' + str(e))
        return resultats

    tree = html.fromstring(reponse.content)
    tous_les_liens = tree.xpath('//a/@href')
    urls_visitees = set()
    compteur = 0

    for href in tous_les_liens:
        if compteur >= nb_articles:
            break
        if not href:
            continue
        if href.startswith('/'):
            href = 'https://www.francetvinfo.fr' + href
        if 'francetvinfo.fr' not in href:
            continue
        section_url = label if label != 'sport' else 'sports'
        if '/' + section_url + '/' not in href:
            continue
        chemins_interdits = ['.jpg', '.png', '.mp4', '/recherche/', '/live/',
                             '/en-direct/', '?', '#', '/tags/', '/videos/']
        if any(c in href for c in chemins_interdits):
            continue
        if not href.endswith('.html'):
            continue
        if href in urls_visitees:
            continue
        urls_visitees.add(href)

        try:
            reponse_article = requests.get(href, headers=HEADERS, timeout=10)
            reponse_article.raise_for_status()
        except:
            continue

        tree_article = html.fromstring(reponse_article.content)
        meta = extraire_metadonnees(tree_article, href)
        texte_final = extraire_texte_article(tree_article)

        if len(texte_final) < 300:
            continue

        resultats.append({
            'url':     href,
            'titre':   meta['titre'],
            'date':    meta['date'],
            'chapeau': meta['chapeau'],
            'texte':   texte_final,
            'label':   label
        })
        print('[' + str(compteur+1) + '/' + str(nb_articles) + '] [' + label + '] ' + meta['titre'][:60])
        compteur += 1
        time.sleep(1)

    return resultats

print('Fonction scraper_section definie.')

In [ ]:
def sauvegarder_csv(donnees, nom_fichier):
    """
    Sauvegarde les articles dans un fichier CSV.
    Entree : donnees (list of dict), nom_fichier (str)
    Sortie : fichier CSV sur le disque
    """
    fichier_existe = os.path.exists(nom_fichier)
    with open(nom_fichier, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f, fieldnames=['url', 'titre', 'date', 'chapeau', 'texte', 'label']
        )
        if not fichier_existe:
            writer.writeheader()
        for article in donnees:
            writer.writerow(article)
    print(str(len(donnees)) + ' articles sauvegardes dans ' + nom_fichier)

print('Fonction sauvegarder_csv definie.')

In [ ]:
# Lancement du scraping
if os.path.exists(FICHIER_CSV):
    os.remove(FICHIER_CSV)
    print('Ancien fichier supprime.')

for label, url in SECTIONS.items():
    print('\n--- Section : ' + label + ' ---')
    articles = scraper_section(url, label, NB_ARTICLES)
    if articles:
        sauvegarder_csv(articles, FICHIER_CSV)
    else:
        print('Aucun article recupere pour : ' + label)

print('\nScraping termine !')

In [ ]:
# Verification du dataset
df = pd.read_csv(FICHIER_CSV)
print('Nombre articles : ' + str(len(df)))
print('\nRepartition par label :')
print(df['label'].value_counts())
print('\nLongueur moyenne textes : ' + str(int(df['texte'].str.len().mean())) + ' caracteres')
print('\nApercu texte premier article (300 car.) :')
print(df['texte'].iloc[0][:300])
df[['titre', 'date', 'label']].head(10)